In [1]:


import os
import glob
import sys
from pathlib import Path

sys.path.append("../..")

import matplotlib.pyplot as plt
import numpy as np
import csv
import torch

from tqdm import tqdm

import nibabel as nib
import json
import argparse

from typing import Union

from utils.utils import dtprint

from monai.metrics import DiceMetric


from scipy.ndimage import median_filter, binary_erosion, binary_dilation



In [2]:
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"
#ROOT_DIR = "/home/fehrdelt/bettik/"

In [3]:
config_dict = json.load(open(ROOT_DIR+"AnoDiffExperiments/experiment_2/exp_2_4/config.json", "r"))
args = argparse.Namespace(**config_dict)


In [4]:
patch_anomaly_maps_folder = ROOT_DIR + "datasets/anomaly_maps/exp_2_4/large/"
slice_anomaly_maps_folder = ROOT_DIR + "datasets/anomaly_maps/exp_2_2/large/"

masks_folder = ROOT_DIR + "datasets/final_soop_dataset_small/masks_combined_registered/"

In [18]:
images_to_analyze = ["sub-3", "sub-8", "sub-53", "sub-56"]

In [8]:
print("Patch DDPM")

Patch DDPM


In [33]:
# Get sorted list of anomaly map files
#patch_anomaly_map_files = [a for a in sorted(glob.glob(patch_anomaly_maps_folder + "*.nii.gz")) if os.path.basename(a).replace('_t_110.nii.gz', '') in images_to_analyze]
patch_anomaly_map_files = sorted(glob.glob(patch_anomaly_maps_folder + "*.nii.gz"))
print(len(patch_anomaly_map_files))

patch_threshold = 0.04
dice_metric = DiceMetric(include_background=False, reduction="mean")

58


In [34]:
print(patch_anomaly_map_files)

['/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-303_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-321_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-338_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-359_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-360_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-366_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-370_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-374_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-398_t_110.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_2_4/large/sub-3_t_110.nii.gz', '/bettik/PR

In [36]:
print("Patch ddpm")

patch_dice_scores = []

for anomaly_map_file in tqdm(patch_anomaly_map_files):
    # Load anomaly map
    anomaly_map = nib.load(anomaly_map_file).get_fdata()
    
    # Apply median filter
    anomaly_map = median_filter(anomaly_map, size=7)
    
    # Get corresponding mask file (assuming same filename)
    subject_id = (Path(anomaly_map_file).name).replace("_t_110", "")
    mask_file = masks_folder + subject_id
    mask = nib.load(mask_file).get_fdata()
    
    # Apply threshold to anomaly map
    thresholded_map = (anomaly_map >= patch_threshold).astype(np.float32)

    # Apply erosion and dilation to the thresholded map
    thresholded_map = binary_erosion(thresholded_map, iterations=6).astype(np.float32)
    thresholded_map = binary_dilation(thresholded_map, iterations=6).astype(np.float32)

    # Apply binary fill holes to the thresholded map
    thresholded_map = binary_fill_holes(thresholded_map).astype(np.float32)

    # Convert to tensors with batch and channel dimensions [B, C, H, W, D]
    pred_tensor = torch.tensor(thresholded_map).unsqueeze(0).unsqueeze(0)
    mask_tensor = torch.tensor(mask).unsqueeze(0).unsqueeze(0)
    
    # Compute Dice score
    dice_metric(y_pred=pred_tensor, y=mask_tensor)
    patch_dice_score = dice_metric.aggregate().item()
    dice_metric.reset()
    
    patch_dice_scores.append(patch_dice_score)
    #print(f"Subject: {subject_id} | Dice Score: {patch_dice_score:.4f}")

print(f"\nOverall Mean Dice Score: {np.mean(patch_dice_scores):.4f} ± {np.std(patch_dice_scores):.4f}")

Patch ddpm


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 58/58 [10:57<00:00, 11.34s/it]


Overall Mean Dice Score: 0.5662 ± 0.2309


In [37]:
# Get sorted list of anomaly map files
#slice_anomaly_map_files = [a for a in sorted(glob.glob(slice_anomaly_maps_folder + "*.nii.gz")) if os.path.basename(a).replace('.nii.gz', '') in images_to_analyze]
slice_anomaly_map_files = sorted(glob.glob(slice_anomaly_maps_folder + "*.nii.gz"))

slice_threshold = 0.04
dice_metric = DiceMetric(include_background=False, reduction="mean")

In [38]:
patch_dice_scores = []

for anomaly_map_file in tqdm(slice_anomaly_map_files):
    # Load anomaly map
    anomaly_map = nib.load(anomaly_map_file).get_fdata()
    
    # Apply median filter
    anomaly_map = median_filter(anomaly_map, size=7)
    
    # Get corresponding mask file (assuming same filename)
    subject_id = (Path(anomaly_map_file).name).replace("_t_110", "")
    mask_file = masks_folder + subject_id
    mask = nib.load(mask_file).get_fdata()
    
    # Apply threshold to anomaly map
    thresholded_map = (anomaly_map >= patch_threshold).astype(np.float32)

    # Apply erosion and dilation to the thresholded map
    thresholded_map = binary_erosion(thresholded_map, iterations=6).astype(np.float32)
    thresholded_map = binary_dilation(thresholded_map, iterations=6).astype(np.float32)

    # Apply binary fill holes to the thresholded map
    thresholded_map = binary_fill_holes(thresholded_map).astype(np.float32)
    
    # Convert to tensors with batch and channel dimensions [B, C, H, W, D]
    pred_tensor = torch.tensor(thresholded_map).unsqueeze(0).unsqueeze(0)
    mask_tensor = torch.tensor(mask).unsqueeze(0).unsqueeze(0)
    
    # Compute Dice score
    dice_metric(y_pred=pred_tensor, y=mask_tensor)
    slice_dice_score = dice_metric.aggregate().item()
    dice_metric.reset()
    
    #print(f"Subject: {subject_id} | Dice Score: {slice_dice_score:.4f}")
    patch_dice_scores.append(slice_dice_score)

print(f"\nOverall Mean Dice Score: {np.mean(patch_dice_scores):.4f} ± {np.std(patch_dice_scores):.4f}")


Overall Mean Dice Score: 0.6122 ± 0.1819
